# Label Merged Dataset

**Function**: Apply terrain/surface labels to the synchronized and resampled sensor data from `merged_dataset.csv`.

## Strategy

1. Load `merged_dataset.csv` from `data/interim/merged/`
2. Split by `run_id` to process each run individually
3. Discover label configurations from raw data directories
4. Apply labels to each run's sensor data
5. Save all labeled data to a single `labeled_dataset.csv` file
6. Generate labeled sensor plots for each run

**Input**: `data/resampled/merged_dataset.csv`  
**Output**: 
- Labeled dataset: `data/interim/labeled/labeled_dataset.csv`
- Plots: `reports/labeled/{run_id}/`


In [1]:
from pathlib import Path
import sys
import pandas as pd

# Make project root importable whether CWD is repo root or notebooks/
cwd = Path.cwd()
PROJECT_ROOT = cwd if (cwd / 'src').exists() else cwd.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.io import load_run, save_labeled_run
from src.labels import (
    load_label_config, 
    discover_labeled_runs, 
    label_dataframe,
    validate_label_transitions, 
    plot_labeled_sensors, 
    plot_labeled_acc, 
    plot_labeled_gyro, 
    plot_labeled_odo, 
    save_labeled_plot
)


In [2]:
# Load merged dataset from resampled data
merged_dataset_path = PROJECT_ROOT / "data" / "interim" / "merged" / "merged_dataset.csv"

if not merged_dataset_path.exists():
    raise FileNotFoundError(f"Merged dataset not found: {merged_dataset_path}\nPlease run 02_sync_resample.ipynb first.")

merged_df = pd.read_csv(merged_dataset_path)

print(f"Loaded merged dataset: {merged_dataset_path}")
print(f"Shape: {merged_df.shape}")
print(f"Columns: {list(merged_df.columns)}")
print(f"Unique run_ids: {merged_df['run_id'].nunique()}")
print(f"\nRun IDs:")
for run_id in sorted(merged_df['run_id'].unique()):
    n_samples = len(merged_df[merged_df['run_id'] == run_id])
    print(f"  - {run_id}: {n_samples} samples")


Loaded merged dataset: /Users/pratyush/Desktop/DTU/Bachelor_thesis/BSC_Thesis_intrinsic_sensor_analysis/data/interim/merged/merged_dataset.csv
Shape: (871377, 11)
Columns: ['t', 't_rel', 'run_id', 'ax', 'ay', 'az', 'gx', 'gy', 'gz', 'v1', 'v2']
Unique run_ids: 3

Run IDs:
  - log_20260223_142511.490: 292115 samples
  - log_20260226_102148.990: 219335 samples
  - log_20260309_141435.414: 359927 samples


In [3]:
# Discover label configurations and match with runs in merged dataset
raw_data_dir = PROJECT_ROOT / "data" / "raw"
runs_with_labels = discover_labeled_runs(raw_data_dir)

print(f"Found {len(runs_with_labels)} run(s) with label config(s):")
for run_dir, config_path in runs_with_labels.items():
    print(f"  - {run_dir.name} -> {config_path.relative_to(PROJECT_ROOT)}")

# Split merged dataset by run_id and match with label configs
run_data_dict = {}
for run_id in merged_df['run_id'].unique():
    # Find label config for this run_id
    label_config_path = None
    for run_dir, config_path in runs_with_labels.items():
        if run_dir.name == run_id:
            label_config_path = config_path
            break
    
    if label_config_path:
        run_subset = merged_df[merged_df['run_id'] == run_id].copy().reset_index(drop=True)
        label_configs = load_label_config(label_config_path)
        
        run_data_dict[run_id] = {
            'data': run_subset,
            'label_configs': label_configs,
            'config_path': label_config_path
        }
        
        print(f"\n{run_id}:")
        print(f"  Label config: {label_config_path.relative_to(PROJECT_ROOT)}")
        print(f"  Labels: {', '.join(label_configs.keys())}")
        print(f"  Samples: {len(run_subset)}")
    else:
        print(f"\n⚠ Warning: No label config found for {run_id}")

print(f"\nTotal runs with labels: {len(run_data_dict)}")


Found 3 run(s) with label config(s):
  - log_20260223_142511.490 -> data/raw/Main/log_20260223_142511.490/labels_config.json
  - log_20260226_102148.990 -> data/raw/Main/log_20260226_102148.990/labels_config.json
  - log_20260309_141435.414 -> data/raw/Main/log_20260309_141435.414/labels_config.json

log_20260223_142511.490:
  Label config: data/raw/Main/log_20260223_142511.490/labels_config.json
  Labels: grass, smooth_terrain, muddy_dirt_track
  Samples: 292115

log_20260226_102148.990:
  Label config: data/raw/Main/log_20260226_102148.990/labels_config.json
  Labels: grass, smooth_terrain, muddy_dirt_track
  Samples: 219335

log_20260309_141435.414:
  Label config: data/raw/Main/log_20260309_141435.414/labels_config.json
  Labels: grass, smooth_terrain, dry_dirt_track
  Samples: 359927

Total runs with labels: 3


In [4]:
# Apply labels to each run's merged data
labeled_runs = {}

for run_id, run_info in run_data_dict.items():
    merged_run_df = run_info['data']
    label_configs = run_info['label_configs']
    
    # Apply labels using the 't' (Unix timestamp) column
    labeled_df = label_dataframe(merged_run_df, label_configs, time_column='t')
    labeled_runs[run_id] = labeled_df
    
    # Show label distribution
    label_counts = labeled_df['label'].value_counts(dropna=False)
    print(f"\n{run_id.upper()} - Label Distribution:")
    print(f"  Total samples: {len(labeled_df)}")
    for label, count in label_counts.items():
        percentage = (count / len(labeled_df)) * 100
        label_name = label if pd.notna(label) else "unlabeled"
        print(f"  {label_name}: {count} samples ({percentage:.1f}%)")



LOG_20260223_142511.490 - Label Distribution:
  Total samples: 292115
  unlabeled: 252615 samples (86.5%)
  smooth_terrain: 18700 samples (6.4%)
  muddy_dirt_track: 13800 samples (4.7%)
  grass: 7000 samples (2.4%)

LOG_20260226_102148.990 - Label Distribution:
  Total samples: 219335
  unlabeled: 201335 samples (91.8%)
  grass: 6000 samples (2.7%)
  smooth_terrain: 6000 samples (2.7%)
  muddy_dirt_track: 6000 samples (2.7%)

LOG_20260309_141435.414 - Label Distribution:
  Total samples: 359927
  unlabeled: 194127 samples (53.9%)
  dry_dirt_track: 83300 samples (23.1%)
  grass: 46800 samples (13.0%)
  smooth_terrain: 35700 samples (9.9%)


In [5]:
# Validate label transitions for each run
for run_id, labeled_df in labeled_runs.items():
    print(f"\n{'='*60}")
    print(f"Validation: {run_id}")
    print('='*60)
    
    # Validate transitions
    validation = validate_label_transitions(labeled_df, max_transitions=3)
    
    print(f"Total transitions: {validation['total_transitions']}")
    for transition in validation['transition_samples']:
        print(f"\nTransition {transition['transition_num']} (row {transition['row_idx']}):")
        print(transition['samples'])



Validation: log_20260223_142511.490
Total transitions: 252617

Transition 1 (row 1):
              t label
0  1.771853e+09   NaN
1  1.771853e+09   NaN
2  1.771853e+09   NaN
3  1.771853e+09   NaN

Transition 2 (row 2):
              t label
0  1.771853e+09   NaN
1  1.771853e+09   NaN
2  1.771853e+09   NaN
3  1.771853e+09   NaN
4  1.771853e+09   NaN

Transition 3 (row 3):
              t label
1  1.771853e+09   NaN
2  1.771853e+09   NaN
3  1.771853e+09   NaN
4  1.771853e+09   NaN
5  1.771853e+09   NaN

Validation: log_20260226_102148.990
Total transitions: 201337

Transition 1 (row 1):
              t label
0  1.772098e+09   NaN
1  1.772098e+09   NaN
2  1.772098e+09   NaN
3  1.772098e+09   NaN

Transition 2 (row 2):
              t label
0  1.772098e+09   NaN
1  1.772098e+09   NaN
2  1.772098e+09   NaN
3  1.772098e+09   NaN
4  1.772098e+09   NaN

Transition 3 (row 3):
              t label
1  1.772098e+09   NaN
2  1.772098e+09   NaN
3  1.772098e+09   NaN
4  1.772098e+09   NaN
5  1.77209

In [6]:
# Save all labeled runs into a single labeled dataset
output_root = PROJECT_ROOT / "data" / "interim" / "labeled"
output_root.mkdir(parents=True, exist_ok=True)

# Concatenate all labeled runs
all_labeled_data = []
for run_id, labeled_df in labeled_runs.items():
    all_labeled_data.append(labeled_df)

labeled_dataset = pd.concat(all_labeled_data, ignore_index=True)

# Save to single CSV file
labeled_output_path = output_root / "labeled_dataset.csv"
labeled_dataset.to_csv(labeled_output_path, index=False)

print(f"\nSaved labeled dataset to: {labeled_output_path}")
print(f"Total shape: {labeled_dataset.shape}")
print(f"Columns: {list(labeled_dataset.columns)}")
print(f"\nLabel distribution:")
label_counts = labeled_dataset['label'].value_counts(dropna=False)
for label, count in label_counts.items():
    percentage = (count / len(labeled_dataset)) * 100
    label_name = label if pd.notna(label) else "unlabeled"
    print(f"  {label_name}: {count} samples ({percentage:.1f}%)")

# Display summary per run
print(f"\nSamples per run:")
for run_id in labeled_dataset['run_id'].unique():
    n_samples = len(labeled_dataset[labeled_dataset['run_id'] == run_id])
    print(f"  {run_id}: {n_samples} samples")

# Display first few rows
labeled_dataset.head()



Saved labeled dataset to: /Users/pratyush/Desktop/DTU/Bachelor_thesis/BSC_Thesis_intrinsic_sensor_analysis/data/interim/labeled/labeled_dataset.csv
Total shape: (871377, 12)
Columns: ['t', 't_rel', 'run_id', 'ax', 'ay', 'az', 'gx', 'gy', 'gz', 'v1', 'v2', 'label']

Label distribution:
  unlabeled: 648077 samples (74.4%)
  dry_dirt_track: 83300 samples (9.6%)
  smooth_terrain: 60400 samples (6.9%)
  grass: 59800 samples (6.9%)
  muddy_dirt_track: 19800 samples (2.3%)

Samples per run:
  log_20260223_142511.490: 292115 samples
  log_20260226_102148.990: 219335 samples
  log_20260309_141435.414: 359927 samples


,t,t_rel,run_id,ax,ay,az,gx,gy,gz,v1,v2,label
0,1.771853e+09,0.00,log_20260223_142511.490,0.062400,-1.002400,0.005600,-3.282700,-9.213900,-1.776900,-0.0,0.0,NaN
1,1.771853e+09,0.01,log_20260223_142511.490,0.057430,-1.002640,0.006321,-3.549665,-9.377383,-1.622703,0.0,0.0,NaN
2,1.771853e+09,0.02,log_20260223_142511.490,0.059592,-1.001632,0.007254,-3.512238,-9.110857,-1.608384,0.0,0.0,NaN
3,1.771853e+09,0.03,log_20260223_142511.490,0.062838,-1.001000,0.008800,-3.530830,-9.084230,-1.570070,0.0,0.0,NaN
4,1.771853e+09,0.04,log_20260223_142511.490,0.062976,-1.000559,0.009747,-3.586942,-9.129320,-1.603502,0.0,0.0,NaN


In [7]:
# Generate and save combined labeled plots for all runs
import matplotlib.pyplot as plt

plots_output_root = PROJECT_ROOT / "reports" / "labeled"

for run_id, labeled_df in labeled_runs.items():
    # Check if combined plot already exists
    run_plot_dir = plots_output_root / run_id
    combined_plot_path = run_plot_dir / "labeled_sensors_plot.png"
    
    if combined_plot_path.exists():
        print(f"\n⏭ Skipping {run_id}: combined plot already exists")
        print(f"  {combined_plot_path.relative_to(PROJECT_ROOT)}")
        continue
    
    # Split into sensor DataFrames for plotting
    acc_labeled = labeled_df[['t', 't_rel', 'ax', 'ay', 'az', 'label']].copy()
    gyro_labeled = labeled_df[['t', 't_rel', 'gx', 'gy', 'gz', 'label']].copy()
    odo_labeled = labeled_df[['t', 't_rel', 'v1', 'v2', 'label']].copy()
    
    # Create combined labeled plot
    fig = plot_labeled_sensors(
        acc=acc_labeled,
        gyro=gyro_labeled,
        odo=odo_labeled,
        tcol='t_rel',
        show=False
    )
    
    # Save plot to reports/labeled/{run_id}/
    plot_path = save_labeled_plot(
        fig,
        run_id=run_id,
        output_root=plots_output_root,
        filename="labeled_sensors_plot.png",
        dpi=150
    )
    
    plt.close(fig)  # Close figure to free memory
    
    print(f"\n✓ Saved combined plot for {run_id}:")
    print(f"  {plot_path.relative_to(PROJECT_ROOT)}")



✓ Saved combined plot for log_20260223_142511.490:
  reports/labeled/log_20260223_142511.490/labeled_sensors_plot.png

✓ Saved combined plot for log_20260226_102148.990:
  reports/labeled/log_20260226_102148.990/labeled_sensors_plot.png

✓ Saved combined plot for log_20260309_141435.414:
  reports/labeled/log_20260309_141435.414/labeled_sensors_plot.png


In [8]:
# Generate and save individual sensor plots (accelerometer, gyroscope, odometry)
import matplotlib.pyplot as plt

plots_output_root = PROJECT_ROOT / "reports" / "labeled"

for run_id, labeled_df in labeled_runs.items():
    # Check if all individual plots already exist
    run_plot_dir = plots_output_root / run_id
    acc_plot_path = run_plot_dir / "labeled_acc_plot.png"
    gyro_plot_path = run_plot_dir / "labeled_gyro_plot.png"
    odo_plot_path = run_plot_dir / "labeled_odo_plot.png"
    
    all_exist = acc_plot_path.exists() and gyro_plot_path.exists() and odo_plot_path.exists()
    
    if all_exist:
        print(f"\n⏭ Skipping {run_id}: all individual plots already exist")
        continue
    
    print(f"\n{'='*60}")
    print(f"Saving individual plots for {run_id}")
    print('='*60)
    
    # Split into sensor DataFrames for plotting
    acc_labeled = labeled_df[['t', 't_rel', 'ax', 'ay', 'az', 'label']].copy()
    gyro_labeled = labeled_df[['t', 't_rel', 'gx', 'gy', 'gz', 'label']].copy()
    odo_labeled = labeled_df[['t', 't_rel', 'v1', 'v2', 'label']].copy()
    
    # Plot and save accelerometer (if not exists)
    if not acc_plot_path.exists():
        fig_acc = plot_labeled_acc(
            acc=acc_labeled,
            tcol='t_rel',
            show=False
        )
        plot_path_acc = save_labeled_plot(
            fig_acc,
            run_id=run_id,
            output_root=plots_output_root,
            filename="labeled_acc_plot.png",
            dpi=150
        )
        plt.close(fig_acc)
        print(f"✓ Saved accelerometer plot:")
        print(f"  {plot_path_acc.relative_to(PROJECT_ROOT)}")
    else:
        print(f"⏭ Accelerometer plot exists, skipping")
    
    # Plot and save gyroscope (if not exists)
    if not gyro_plot_path.exists():
        fig_gyro = plot_labeled_gyro(
            gyro=gyro_labeled,
            tcol='t_rel',
            show=False
        )
        plot_path_gyro = save_labeled_plot(
            fig_gyro,
            run_id=run_id,
            output_root=plots_output_root,
            filename="labeled_gyro_plot.png",
            dpi=150
        )
        plt.close(fig_gyro)
        print(f"✓ Saved gyroscope plot:")
        print(f"  {plot_path_gyro.relative_to(PROJECT_ROOT)}")
    else:
        print(f"⏭ Gyroscope plot exists, skipping")
    
    # Plot and save odometry (if not exists)
    if not odo_plot_path.exists():
        fig_odo = plot_labeled_odo(
            odo=odo_labeled,
            tcol='t_rel',
            show=False
        )
        plot_path_odo = save_labeled_plot(
            fig_odo,
            run_id=run_id,
            output_root=plots_output_root,
            filename="labeled_odo_plot.png",
            dpi=150
        )
        plt.close(fig_odo)
        print(f"✓ Saved odometry plot:")
        print(f"  {plot_path_odo.relative_to(PROJECT_ROOT)}")
    else:
        print(f"⏭ Odometry plot exists, skipping")



Saving individual plots for log_20260223_142511.490
✓ Saved accelerometer plot:
  reports/labeled/log_20260223_142511.490/labeled_acc_plot.png
✓ Saved gyroscope plot:
  reports/labeled/log_20260223_142511.490/labeled_gyro_plot.png
✓ Saved odometry plot:
  reports/labeled/log_20260223_142511.490/labeled_odo_plot.png

Saving individual plots for log_20260226_102148.990
✓ Saved accelerometer plot:
  reports/labeled/log_20260226_102148.990/labeled_acc_plot.png
✓ Saved gyroscope plot:
  reports/labeled/log_20260226_102148.990/labeled_gyro_plot.png
✓ Saved odometry plot:
  reports/labeled/log_20260226_102148.990/labeled_odo_plot.png

Saving individual plots for log_20260309_141435.414
✓ Saved accelerometer plot:
  reports/labeled/log_20260309_141435.414/labeled_acc_plot.png
✓ Saved gyroscope plot:
  reports/labeled/log_20260309_141435.414/labeled_gyro_plot.png
✓ Saved odometry plot:
  reports/labeled/log_20260309_141435.414/labeled_odo_plot.png
